# Week 3 Validation and First KPIs

This notebook validates the processed warehouse data and calculates the first business KPIs.

## 1. Load the processed data

The processed sales and purchases fact tables are the main inputs for this analysis. The detailed payments file is used only for the individual payment-date check because that detail is not retained in the processed sales table.

In [30]:
from pathlib import Path
import pandas as pd

root = Path('../../')
processed = root / 'data' / 'processed'
clean = root / 'data' / 'cleaned'

sales_table = pd.read_csv(processed / 'sales_table.csv')
purchases_fact_table = pd.read_csv(processed / 'purchases_fact_table.csv')
invoices = pd.read_csv(clean / 'invoices.csv')
payments = pd.read_csv(clean / 'payments.csv')

sales_date_columns = [
    'order_date',
    'delivery_date',
    'invoice_date',
    'last_payment_date',
]
for column in sales_date_columns:
    sales_table[column] = pd.to_datetime(sales_table[column], errors='coerce')

invoices['invoice_date'] = pd.to_datetime(invoices['invoice_date'], errors='coerce')
purchases_fact_table['movement_date'] = pd.to_datetime(
    purchases_fact_table['movement_date'], errors='coerce'
)
payments['payment_date'] = pd.to_datetime(payments['payment_date'], errors='coerce')

invoice_table = sales_table.dropna(subset=['invoice_id']).drop_duplicates('invoice_id')
order_table = sales_table.drop_duplicates('so_id')

print('Processed sales rows:', len(sales_table))
print('Processed purchase rows:', len(purchases_fact_table))
print('Unique invoices:', len(invoice_table))
print('Unique orders:', len(order_table))

Processed sales rows: 130402
Processed purchase rows: 237230
Unique invoices: 17836
Unique orders: 20000


## 2. Payment amounts versus invoice totals

Payments are first summarised to one row per invoice. This avoids counting the same payment once for every sales line.

In [31]:
invoice_check = invoice_table[
    ['invoice_id', 'grand_total', 'total_paid', 'payment_status']
].copy()
invoice_check['total_paid'] = invoice_check['total_paid'].fillna(0)
invoice_check['outstanding'] = invoice_check['grand_total'] - invoice_check['total_paid']

overpaid = invoice_check[invoice_check['total_paid'] > invoice_check['grand_total']]
overpaid_with_tolerance = invoice_check[
    invoice_check['total_paid'] > invoice_check['grand_total'] + 0.01
]

print('Invoices with total paid above invoice total:', len(overpaid))
print('Invoices overpaid by more than 0.01:', len(overpaid_with_tolerance))
print('Invoices with positive outstanding balance:', (invoice_check['outstanding'] > 0).sum())

Invoices with total paid above invoice total: 308
Invoices overpaid by more than 0.01: 161
Invoices with positive outstanding balance: 5586


## 3. Orphan record checks

In [32]:
sales_required = ['so_id', 'product_id', 'customer_id', 'branch_id']
purchase_required = ['product_id', 'branch_id', 'movement_id']

missing_sales_links = sales_table[sales_table[sales_required].isna().any(axis=1)]
missing_purchase_links = purchases_fact_table[
    purchases_fact_table[purchase_required].isna().any(axis=1)
]

print('Processed sales rows missing linked records:', len(missing_sales_links))
print('Processed purchase rows missing linked records:', len(missing_purchase_links))

Processed sales rows missing linked records: 0
Processed purchase rows missing linked records: 0


## 4. Negative value checks

In [33]:
negative_checks = {
    'negative sales quantities': (sales_table['quantity'] < 0).sum(),
    'negative sales prices': (sales_table['unit_price'] < 0).sum(),
    'negative payment totals': (sales_table['total_paid'].fillna(0) < 0).sum(),
    'negative stock quantities': (purchases_fact_table['quantity'] < 0).sum(),
    'negative current stock': (purchases_fact_table['current_stock'] < 0).sum(),
}
for check, count in negative_checks.items():
    print(f'{check}: {count}')

negative sales quantities: 0
negative sales prices: 0
negative payment totals: 0
negative stock quantities: 0
negative current stock: 0


## 5. Date logic checks

In [34]:
invoice_dates = sales_table[
    ['so_id', 'order_date', 'invoice_date']
].drop_duplicates('so_id')
payment_dates = payments.merge(
    invoices[['invoice_id', 'invoice_date']],
    on='invoice_id',
    how='left',
)

date_checks = {
    'orders delivered before order date': (
        sales_table['delivery_date'] < sales_table['order_date']
    ).sum(),
    'invoices before order date': (
        invoice_dates['invoice_date'] < invoice_dates['order_date']
    ).sum(),
    'payments before invoice date': (
        payment_dates['payment_date'] < payment_dates['invoice_date']
    ).sum(),
}
for check, count in date_checks.items():
    print(f'{check}: {count}')

orders delivered before order date: 0
invoices before order date: 0
payments before invoice date: 208


## 6. Stock ledger running-balance check

Only movements with an unambiguous order are checked. Product, branch, and movement-date groups with more than one movement are excluded because the exact same-day order is unknown. Adjustment movements are excluded because they do not have a simple IN or OUT sign.

In [35]:
stock_check = purchases_fact_table.copy()
stock_check = stock_check[stock_check['movement_type'].isin(['IN', 'OUT'])].copy()

daily_counts = (
    stock_check.groupby(['product_id', 'branch_id', 'movement_date'])
    .size()
    .rename('movement_count')
    .reset_index()
)
stock_check = stock_check.merge(
    daily_counts,
    on=['product_id', 'branch_id', 'movement_date'],
)
stock_check = stock_check[stock_check['movement_count'] == 1].copy()
stock_check = stock_check.sort_values(
    ['product_id', 'branch_id', 'movement_date']
)

stock_check['previous_balance'] = stock_check.groupby(
    ['product_id', 'branch_id']
)['running_balance'].shift(1)
stock_check['movement_sign'] = stock_check['movement_type'].map({'IN': 1, 'OUT': -1})
stock_check['expected_balance'] = (
    stock_check['previous_balance']
    + stock_check['movement_sign'] * stock_check['quantity']
)
checked_movements = stock_check.dropna(subset=['previous_balance'])
balance_errors = checked_movements[
    checked_movements['running_balance'] != checked_movements['expected_balance']
]

error_rate = len(balance_errors) / len(checked_movements) if len(checked_movements) else 0
print('Unambiguous movements checked:', len(checked_movements))
print('Running-balance errors:', len(balance_errors))
print(f'Running-balance error rate: {error_rate:.2%}')
print('Unmatched processed purchase rows:', len(purchases_fact_table) - len(stock_check))

Unambiguous movements checked: 111144
Running-balance errors: 36604
Running-balance error rate: 32.93%
Unmatched processed purchase rows: 125906


## 7. First KPIs

In [36]:
order_totals = order_table[['so_id', 'grand_total']]
total_revenue = order_totals['grand_total'].sum()
order_count = order_totals['so_id'].nunique()
average_order_value = total_revenue / order_count
outstanding_invoices = invoice_check[invoice_check['outstanding'] > 0]
outstanding_payments = outstanding_invoices['outstanding'].sum()
active_customers = order_table['customer_id'].nunique()

inventory_values = purchases_fact_table[
    ['product_id', 'branch_id', 'current_stock', 'unit_cost']
].drop_duplicates(['product_id', 'branch_id'])
total_stock_value = (
    inventory_values['current_stock'] * inventory_values['unit_cost']
).sum()

supplier_lead_time = purchases_fact_table[
    ['product_id', 'lead_time_days']
].drop_duplicates('product_id')['lead_time_days']
average_supplier_lead_time = supplier_lead_time.mean()

kpis = {
    'Total sales revenue': total_revenue,
    'Total number of orders': order_count,
    'Average order value': average_order_value,
    'Outstanding payments': outstanding_payments,
    'Invoices with outstanding balance': len(outstanding_invoices),
    'Active customers': active_customers,
    'Total stock value': total_stock_value,
    'Average supplier lead time (days)': average_supplier_lead_time,
}

for name, value in kpis.items():
    if isinstance(value, float):
        print(f'{name}: {value:,.2f}')
    else:
        print(f'{name}: {value:,}')

Total sales revenue: 32,424,754,555.80
Total number of orders: 20,000
Average order value: 1,621,237.73
Outstanding payments: 5,835,028,760.19
Invoices with outstanding balance: 5,586
Active customers: 500
Total stock value: 281,292,677,410
Average supplier lead time (days): 17.00


## Summary

The checks above preserve questionable records for investigation rather than deleting them. The main issues to investigate later are overpaid invoices, payments dated before their invoices, and stock ledger movements that do not reconcile.

## Reconciliation note

This direct check gives a 32.93% running-balance error rate for the unambiguous IN/OUT movements. The earlier profiling report recorded about 10%, so the two methods should be reconciled before the final report is treated as authoritative. The source ledger is preserved unchanged.